# MIND-small: User-based collaborative filtering (UBCF)

Train a user–article neighborhood recommender from observed clicks. Score original candidate slates using cosine similarity between users, with popularity fallback. Use only training feedback for held-out partitions; train impressions use strictly earlier training days.

## 1. Setup

Read the canonical preprocessing handoff. Run `notebooks/01_Data_Preprocessing.ipynb` first if the processed files are absent. No separate feature-engineering exports are required for this collaborative-filtering baseline.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors


def find_root():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for p in options:
        if (p / 'src' / 'preprocessing.py').is_file():
            return p.resolve()
    raise FileNotFoundError('Run from the extracted bt4222-main repository, or set BT4222_PROJECT_ROOT.')

ROOT = find_root()
PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', ROOT / 'data/processed/v2')).resolve()
if not (PROCESSED_DIR / 'manifest.json').exists():
    alternatives = [ROOT / 'data/processed/v1', ROOT / 'data/processed/v2']
    PROCESSED_DIR = next((p for p in alternatives if (p / 'manifest.json').exists()), PROCESSED_DIR)
if not (PROCESSED_DIR / 'manifest.json').exists():
    raise FileNotFoundError(f'Preprocessing manifest missing: {PROCESSED_DIR}')
OUTPUT_DIR = ROOT / 'outputs/models/ubcf_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_K = 50
CANDIDATE_CHUNK_SIZE = 100_000
print('Processed:', PROCESSED_DIR)
print('Output:', OUTPUT_DIR)

## 2. Load partitions and check impression integrity

Candidate labels are targets, never candidate features. Preserve event keys and candidate positions for impression-level evaluation.

In [ ]:
PARTITIONS = ('train', 'validation', 'test')
def load_partition(part):
    folder = PROCESSED_DIR / part
    events = pd.read_csv(folder / 'events.csv', usecols=['event_key','user_id','timestamp','history_ids'],
                         dtype={'event_key':str,'user_id':str,'history_ids':str})
    events['timestamp'] = pd.to_datetime(events['timestamp'], errors='raise')
    candidates = pd.read_csv(folder / 'candidates.csv',
                             usecols=['event_key','candidate_position','news_id','label'],
                             dtype={'event_key':str,'candidate_position':'int32','news_id':str,'label':'int8'})
    assert events.event_key.is_unique
    assert not candidates.duplicated(['event_key','candidate_position']).any()
    assert candidates.label.isin([0,1]).all()
    assert candidates.event_key.isin(events.event_key).all()
    return events, candidates

data = {part:load_partition(part) for part in PARTITIONS}
for part,(events,candidates) in data.items():
    print(part, 'events:',len(events),'candidates:',len(candidates),
          'dates:',events.timestamp.min(),events.timestamp.max())

## 3. Fit user neighborhoods from permitted earlier clicks

Build a binary user–article interaction matrix from positive outcomes. Cosine similarities are calculated on implicit click histories. Each training day is scored using only earlier training days; validation and test are scored using training clicks only. Similarities exclude the target user.

In [ ]:
train_events, train_candidates = data['train']
train_events = train_events[['event_key','user_id','timestamp']].copy()
train_events['day'] = train_events.timestamp.dt.normalize()
train_clicks = train_candidates.loc[train_candidates.label.eq(1), ['event_key','news_id']].merge(
    train_events, on='event_key', how='left', validate='many_to_one')
assert train_clicks.timestamp.notna().all()

class UBCF:
    def __init__(self, k=50):
        self.k = k

    def fit(self, clicks):
        self.user_ids = pd.Index(sorted(clicks.user_id.unique()))
        self.article_ids = pd.Index(sorted(clicks.news_id.unique()))
        self.user_index = pd.Series(np.arange(len(self.user_ids)), index=self.user_ids)
        self.article_index = pd.Series(np.arange(len(self.article_ids)), index=self.article_ids)
        u = self.user_index.loc[clicks.user_id].to_numpy(dtype=np.int32)
        a = self.article_index.loc[clicks.news_id].to_numpy(dtype=np.int32)
        mat = sparse.csr_matrix((np.ones(len(u),dtype=np.float32),(u,a)),
                                shape=(len(self.user_ids),len(self.article_ids)))
        mat.data[:] = 1.0  # binary implicit feedback; repeated clicks count once
        self.matrix = mat
        self.popularity = np.asarray(mat.sum(axis=0)).ravel().astype(np.float32)
        self.popularity /= max(float(self.popularity.max(initial=0)),1.0)
        self.neighbors = None
        if len(self.user_ids) > 1 and self.matrix.nnz:
            self.neighbors = NearestNeighbors(metric='cosine', algorithm='brute',
                                              n_neighbors=min(self.k+1,len(self.user_ids)),n_jobs=-1)
            self.neighbors.fit(mat)
        return self

    def score_events(self, events, candidates):
        # Group by user to avoid recomputing the same neighborhood for every candidate.
        users = events[['event_key','user_id']].set_index('event_key')['user_id']
        out = np.empty(len(candidates), dtype=np.float32)
        event_keys = candidates.event_key.to_numpy()
        candidate_users = users.reindex(event_keys)
        assert candidate_users.notna().all()
        candidate_users = candidate_users.to_numpy()
        article_pos = self.article_index.reindex(candidates.news_id).to_numpy()
        known_article = pd.notna(article_pos)
        for uid in pd.unique(candidate_users):
            positions = np.flatnonzero(candidate_users == uid)
            known = positions[known_article[positions]]
            out[positions] = 0.0
            if len(known) == 0:
                continue
            cols = article_pos[known].astype(np.int32)
            # Fallback is training-only popularity, including unseen users.
            scores = self.popularity[cols].copy()
            if uid in self.user_index.index and self.neighbors is not None:
                row_id = int(self.user_index.loc[uid])
                distances, indices = self.neighbors.kneighbors(self.matrix[row_id],return_distance=True)
                neighbor_ids = indices[0]
                weights = np.clip(1.0 - distances[0],0,1)
                mask = neighbor_ids != row_id
                neighbor_ids, weights = neighbor_ids[mask], weights[mask]
                valid = weights > 0
                neighbor_ids, weights = neighbor_ids[valid], weights[valid]
                if len(neighbor_ids):
                    neighbor_clicks = self.matrix[neighbor_ids][:,cols]
                    weighted = np.asarray(neighbor_clicks.T @ weights).ravel()
                    scores = (weighted / weights.sum()).astype(np.float32)
            out[known] = scores
        return out

# Synthetic check: shared clicks identify similar users, self is excluded.
toy = pd.DataFrame({'user_id':['u1','u1','u2','u2','u3'],
                    'news_id':['a','b','a','c','d']})
toy_model = UBCF(k=2).fit(toy)
toy_e = pd.DataFrame({'event_key':['e'],'user_id':['u1']})
toy_c = pd.DataFrame({'event_key':['e','e'],'news_id':['c','d']})
assert toy_model.score_events(toy_e,toy_c)[0] > toy_model.score_events(toy_e,toy_c)[1]
print('Synthetic neighbor test passed.')

## 4. Score candidates without future feedback

Training-day scores are strictly out-of-day. The held-out partitions share one model trained on the complete training partition. Outputs retain original candidate order and labels for subsequent evaluation.

In [ ]:
results = {}
for part,(events,candidates) in data.items():
    if part == 'train':
        dates = events.timestamp.dt.normalize()
        scores = np.zeros(len(candidates),dtype=np.float32)
        for day in sorted(dates.unique()):
            event_subset = events.loc[dates.eq(day)]
            selected = candidates.event_key.isin(event_subset.event_key)
            click_subset = train_clicks.loc[train_clicks.day.lt(day)]
            if click_subset.empty:
                # No historical feedback exists for the first training day.
                scores[selected.to_numpy()] = 0.0
            else:
                model = UBCF(TOP_K).fit(click_subset)
                scores[selected.to_numpy()] = model.score_events(event_subset,candidates.loc[selected])
            print('train day:',day,'candidates:',int(selected.sum()),'past clicks:',len(click_subset))
    else:
        if 'heldout_model' not in globals():
            heldout_model = UBCF(TOP_K).fit(train_clicks)
        scores = np.empty(len(candidates),dtype=np.float32)
        for start in range(0,len(candidates),CANDIDATE_CHUNK_SIZE):
            batch = candidates.iloc[start:start+CANDIDATE_CHUNK_SIZE]
            scores[start:start+len(batch)] = heldout_model.score_events(events,batch)
    assert np.isfinite(scores).all() and (scores>=0).all() and (scores<=1.00001).all()
    result = candidates.copy()
    result['ubcf_score'] = scores
    result.to_csv(OUTPUT_DIR / f'{part}_ubcf_scores.csv',index=False)
    results[part] = result
    print(part,'saved:',len(result))

## 5. Evaluate complete impression slates

Use AUC, MRR and nDCG@5/10 on the original candidate lists. Metrics are macro-averaged over eligible impressions; AUC excludes impressions with only one class. No test scores are used to tune `TOP_K`.

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(frame):
    rows=[]
    for _,group in frame.groupby('event_key',sort=False):
        y=group.label.to_numpy(dtype=np.int8)
        s=group.ubcf_score.to_numpy(dtype=np.float64)
        order=np.argsort(-s,kind='stable')
        ranked=y[order]
        positives=int(y.sum())
        if not positives: continue
        ranks=np.flatnonzero(ranked)+1
        first=int(ranks[0])
        row={'MRR':1/first}
        for k in (5,10):
            dcg=sum(float(ranked[i])/np.log2(i+2) for i in range(min(k,len(y))))
            ideal=sum(1/np.log2(i+2) for i in range(min(k,positives)))
            row[f'nDCG@{k}']=dcg/ideal if ideal else np.nan
        row['AUC']=roc_auc_score(y,s) if 0<positives<len(y) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).mean().to_dict(),len(rows)

summary=[]
for part,frame in results.items():
    metrics,n=evaluate(frame)
    summary.append({'partition':part,'eligible_impressions':n,**metrics})
summary=pd.DataFrame(summary)
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR / 'metrics.csv',index=False)

## 6. Verify exports and reproducibility

Check candidate row alignment, finite scores, unchanged labels, and no candidate loss. Run multiple `TOP_K` values on validation only if tuning; freeze the choice before evaluating test.

In [ ]:
for part,(_,original) in data.items():
    exported=pd.read_csv(OUTPUT_DIR / f'{part}_ubcf_scores.csv',
                         dtype={'event_key':str,'news_id':str,'candidate_position':'int32','label':'int8'})
    assert len(exported)==len(original)
    assert exported[['event_key','candidate_position','news_id','label']].equals(
        original[['event_key','candidate_position','news_id','label']])
    assert exported.ubcf_score.between(0,1.00001).all()
    assert exported.ubcf_score.notna().all()
print('All exported partitions verified.')